<a href="https://colab.research.google.com/github/FurkanGozukara/Stable-Diffusion/blob/main/ColabNotebooks/1_click_deep_fake_for_free_by_SECourses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Updated 27 August 2026 - tested on the current Colab GPU runtime
### Fresh setup verified with Python 3.13, Torch 2.11, CUDA 12.8, and a T4 GPU
### Someone upgraded to Gold Tier and I fixed all issues : https://www.patreon.com/c/SECourses
## Most Advanced DeepFake FaceFusion for Windows, RunPod and Massed Compute : https://www.patreon.com/posts/103765029
## Very Advanced VisoMaster for Windows and Massed Compute : https://www.patreon.com/posts/121570322
## Deep Live Cam for Windows : https://www.patreon.com/posts/125826778

## If notebook gets broken get a membership on Patreon and message me for fixing

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import subprocess
import sys

ROOP_DIR = Path('/content/roop')
BASICSR_DIR = ROOP_DIR / 'BasicSR'

def clone_or_update(url, destination, branch):
    destination = Path(destination)
    if (destination / '.git').is_dir():
        print(f'Updating {destination.name}...')
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)
        return
    if destination.exists():
        raise RuntimeError(f'{destination} exists but is not a Git checkout. Rename or remove it, then rerun this cell.')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, url, str(destination)], check=True)

clone_or_update('https://github.com/FurkanGozukara/rop_fixed.git', ROOP_DIR, 'main')
clone_or_update('https://github.com/FurkanGozukara/BasicSR.git', BASICSR_DIR, 'master')

# Install the patched BasicSR checkout as a regular package so it is immediately importable
# in this kernel and satisfies GFPGAN without fetching the incompatible PyPI source.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--force-reinstall', str(BASICSR_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOP_DIR / 'a.txt')], check=True)

import basicsr
import onnxruntime as ort

providers = ort.get_available_providers()
if 'CUDAExecutionProvider' not in providers:
    raise RuntimeError(f'CUDAExecutionProvider is unavailable. Select a GPU runtime and rerun this cell. Providers: {providers}')

print('Setup complete.')
print('Python:', sys.version.split()[0])
print('ONNX Runtime:', ort.__version__)
print('Providers:', providers)
print('TensorFlow:', metadata.version('tensorflow'))
print('BasicSR:', basicsr.__version__)

**Upload the source image and target video to `/content/roop`, or run the optional example-download cell at the bottom first. Then set the three paths in a processing cell.**

**You will see a live `Processing` progress bar. The first run also downloads the InsightFace and GFPGAN model weights.**

**Quality 1 gives the best quality and largest file; quality 100 gives the lowest quality and smallest file. Temporary frames are removed by default to save disk space.**

In [ ]:
from pathlib import Path
import subprocess
import sys

source_path = '/content/roop/face2.png' # @param {"type":"string"}
target_path = '/content/roop/test_video.mp4' # @param {"type":"string"}
output_path = '/content/roop/face_changed_video_v2.mp4' # @param {"type":"string"}
keep_frames = False # @param {"type":"boolean"}

for label, path in [('source', source_path), ('target', target_path)]:
    if not Path(path).is_file():
        raise FileNotFoundError(f'The {label} file does not exist: {path}')

command = [
    sys.executable, '-u', 'run.py',
    '-s', source_path, '-t', target_path, '-o', output_path,
    '--keep-fps', '--temp-frame-quality', '1', '--output-video-quality', '1',
    '--execution-provider', 'cuda',
]
if keep_frames:
    command.append('--keep-frames')
subprocess.run(command, cwd='/content/roop', check=True)
if not Path(output_path).is_file():
    raise RuntimeError(f'Processing finished without creating {output_path}')
print(f'Finished: {output_path}')

**Below code will do also face restoration to improve quality significantly but it will take longer**

In [ ]:
from pathlib import Path
import subprocess
import sys

source_path = '/content/roop/face2.png' # @param {"type":"string"}
target_path = '/content/roop/test_video.mp4' # @param {"type":"string"}
output_path = '/content/roop/face_restored_video3.mp4' # @param {"type":"string"}
keep_frames = False # @param {"type":"boolean"}

for label, path in [('source', source_path), ('target', target_path)]:
    if not Path(path).is_file():
        raise FileNotFoundError(f'The {label} file does not exist: {path}')

command = [
    sys.executable, '-u', 'run.py',
    '-s', source_path, '-t', target_path, '-o', output_path,
    '--keep-fps', '--temp-frame-quality', '1', '--output-video-quality', '1',
    '--execution-provider', 'cuda',
    '--frame-processor', 'face_swapper', 'face_enhancer',
]
if keep_frames:
    command.append('--keep-frames')
subprocess.run(command, cwd='/content/roop', check=True)
if not Path(output_path).is_file():
    raise RuntimeError(f'Processing finished without creating {output_path}')
print(f'Finished: {output_path}')

### All options are displayed below
Append any of them to the above commands before executing
```
python run.py [options]

-h, --help                                                                 show this help message and exit
-s SOURCE_PATH, --source SOURCE_PATH                                       select an source image
-t TARGET_PATH, --target TARGET_PATH                                       select an target image or video
-o OUTPUT_PATH, --output OUTPUT_PATH                                       select output file or directory
--frame-processor FRAME_PROCESSOR [FRAME_PROCESSOR ...]                    frame processors (choices: face_swapper, face_enhancer, ...)
--keep-fps                                                                 keep target fps
--keep-frames                                                              keep temporary frames
--skip-audio                                                               skip target audio
--many-faces                                                               process every face
--reference-face-position REFERENCE_FACE_POSITION                          position of the reference face
--reference-frame-number REFERENCE_FRAME_NUMBER                            number of the reference frame
--similar-face-distance SIMILAR_FACE_DISTANCE                              face distance used for recognition
--temp-frame-format {jpg,png}                                              image format used for frame extraction
--temp-frame-quality [0-100]                                               image quality used for frame extraction
--output-video-encoder {libx264,libx265,libvpx-vp9,h264_nvenc,hevc_nvenc}  encoder used for the output video
--output-video-quality [0-100]                                             quality used for the output video
--max-memory MAX_MEMORY                                                    maximum amount of RAM in GB
--execution-provider {tensorrt,cuda,cpu} [{tensorrt,cuda,cpu} ...]          available execution provider
--execution-threads EXECUTION_THREADS                                      number of execution threads
-v, --version                                                              show program's version number and exit
  ```

### Download a generated output

In [ ]:
from pathlib import Path
from google.colab import files

output_path = '/content/roop/face_changed_video_v2.mp4' # @param {"type":"string"}
output_file = Path(output_path)
if not output_file.is_file():
    raise FileNotFoundError(f'Generated output not found: {output_file}')
files.download(str(output_file))


In [ ]:
# Optional public smoke-test inputs. Run this before a processing cell.
from pathlib import Path
from urllib.request import urlretrieve

examples = {
    'face2.png': 'https://huggingface.co/MonsterMMORPG/examples2/resolve/main/face2.png',
    'test_video.mp4': 'https://huggingface.co/MonsterMMORPG/examples2/resolve/main/test_video.mp4',
}
for filename, url in examples.items():
    destination = Path('/content/roop') / filename
    if destination.is_file():
        print(f'Reusing {destination}')
    else:
        print(f'Downloading {filename}...')
        urlretrieve(url, destination)
print('Example inputs are ready.')